In [3]:
import pyspark
from pyspark import SparkContext

sc = SparkContext.getOrCreate()

data = sc.textFile("work/calidad_aire_datos_meteo_mes.csv")
cabecera = data.first()

seiscuatro = (data
        .filter(lambda linea: linea != cabecera)
        .map(lambda linea: linea.replace(",", "."))
        .map(lambda linea: linea.split(";"))
    
        .filter(lambda campos: campos[0] == "28")   # Madrid
        .filter(lambda campos: campos[3] == "83")   # Magnitud temperatura
        .filter(lambda campos: campos[1] == "6")    # municipio 6
        .filter(lambda campos: campos[2] == "4")    # estación 4
        .map(lambda campos: (                       # seiscuatro[0]->fecha, seiscuatro[1]->suma de temperaturas, seiscuatro[2]->numero de temperaturas
            (campos[5] + "-" + campos[6].zfill(2) + "-" + campos[7].zfill(2)), 
            sum(
                float((campos[i])) 
                for i in range(8, 56, 2)
                if campos[i+1] == "V" and campos[i] != ""
                ),
            sum(
                1
                for i in range(8, 56, 2)
                if campos[i+1] == "V" and campos[i] != ""
                )
            ) 
        )
        .map(lambda campos: (campos[0], campos[1]/campos[2]))
)

cincodos = (data
        .filter(lambda linea: linea != cabecera)
        .map(lambda linea: linea.replace(",", "."))
        .map(lambda linea: linea.split(";"))
    
        .filter(lambda campos: campos[0] == "28")   # Madrid
        .filter(lambda campos: campos[3] == "83")   # Magnitud temperatura
        .filter(lambda campos: campos[1] == "5")    # municipio 5
        .filter(lambda campos: campos[2] == "2")    # estación 2
        .map(lambda campos: (                   
            (campos[5] + "-" + campos[6].zfill(2) + "-" + campos[7].zfill(2)), 
            sum(
                float((campos[i])) 
                for i in range(8, 56, 2)
                if campos[i+1] == "V" and campos[i] != ""
                ),
            sum(
                1
                for i in range(8, 56, 2)
                if campos[i+1] == "V" and campos[i] != ""
                )
            ) 
        )
        .map(lambda campos: (campos[0], campos[1]/campos[2]))
)

union = seiscuatro.join(cincodos) #deberia quedar medias[0] = fecha y medias[1] = media64,media52

porcentajes = (union
        .filter(lambda x: x[1][0] != 0)
        .map(lambda x: (x[0], (x[1][1] / x[1][0])*100))
        .collect()
)

for r in porcentajes:
    print("Fecha:", r[0],
          ", Porcentaje",round(r[1], 2), "%"
          )


Fecha: 2026-02-07 , Porcentaje 131.91 %
Fecha: 2026-02-02 , Porcentaje 113.45 %
Fecha: 2026-02-04 , Porcentaje 114.01 %
Fecha: 2026-02-05 , Porcentaje 112.09 %
Fecha: 2026-02-06 , Porcentaje 116.52 %
Fecha: 2026-02-01 , Porcentaje 129.04 %
Fecha: 2026-02-03 , Porcentaje 114.98 %
Fecha: 2026-02-08 , Porcentaje 116.34 %
